In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

import healpy as hp
from astropy.coordinates import SkyCoord
from astropy import units

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
# mollweide plot setting
# http://balbuceosastropy.blogspot.com/2013/09/the-mollweide-projection.html
def set_mwd(ax, org=0):
    # org is the origin of the plot, 0 or a multiple of 30 degrees in [0,360).
    tick_labels = np.array([150, 120, 90, 60, 30, 0, 330, 300, 270, 240, 210])
    tick_labels = np.remainder(tick_labels+360+org,360)
    ax.set_xticklabels(tick_labels)     # we add the scale on the x axis
    ax.set_xlabel("R.A. [deg]")
    ax.xaxis.label.set_fontsize(12)
    ax.set_ylabel("Dec. [deg]")
    ax.yaxis.label.set_fontsize(12)
    ax.grid(True)
    return True

def get_radec_mw(ras, decs, org):
    ras = np.remainder(ras + 360 - org, 360) # shift ra values
    ras[ras > 180] -= 360 # scale conversion to [-180, 180]
    ras =- ras # reverse the scale: East to the left
    return np.radians(ras), np.radians(decs)

def plot_gp(ax, org=120, c="0.2", lw=0.5, alpha=1, zorder=0):
    tmpn = 1000
    cs = SkyCoord(l = np.linspace(0, 360, tmpn) * units.deg, b = np.zeros(tmpn) * units.deg, frame = "galactic")
    ras, decs = cs.icrs.ra.degree, cs.icrs.dec.degree
    ramws, decmws = get_radec_mw(ras, decs, org)
    ii = ramws.argsort()
    ramws, decmws = ramws[ii], decmws[ii]
    ax.plot(ramws, decmws, c=c, lw=lw, alpha=alpha, zorder=zorder)

In [4]:
target_class = 'LRG'

target_ver_str = '1.0.0'

min_nobs = 1
# maskbits_dict = {'LRG': [1, 8, 9, 11, 12, 13], 'ELG': [1, 11, 12, 13], 'QSO': [1, 8, 9, 11, 12, 13], 'BGS_ANY': [1, 13], 'BGS_BRIGHT': [1, 13]}
maskbits_dict = {'LRG': [], 'ELG': [1, 11, 12, 13], 'QSO': [1, 8, 9, 11, 12, 13], 'BGS_ANY': [1, 13], 'BGS_BRIGHT': [1, 13]}

apply_lrgmask = True
if apply_lrgmask:
    lrgmask_str = '_lrgmask_v1'
else:
    lrgmask_str = ''

randoms_counts_dir = '/Users/rongpu/Documents/Data/desi_targets/dr9.0/imaging_sys/randoms_stats/0.49.0/resolve/counts'
randoms_systematics_dir = '/Users/rongpu/Documents/Data/desi_targets/dr9.0/imaging_sys/randoms_stats/0.49.0/resolve/systematics'
target_densities_dir = '/Users/rongpu/Documents/Data/desi_targets/dr9.0/imaging_sys/density_maps/{}/resolve'.format(target_ver_str)

top_plot_dir = '/Users/rongpu/Documents/Work/DESI/imaging_sys/density_maps/{}/resolve'.format(target_ver_str)

dpi_dict = {64: 200, 128: 200, 256: 600, 512: 1600}
xsize_dict = {64: 8000, 128: 8000, 256: 12000, 512: 16000}
vrange_dict = {'BGS_ANY': {64: [800, 2000], 128: [650, 2150], 256: [400, 2400], 512: [0, 2800]},
               'BGS_BRIGHT': {64: [500, 1200], 128: [350, 1350], 256: [200, 1500], 512: [-200, 1800]},
               'LRG': {64: [300, 900], 128: [200, 1000], 256: [100, 1100], 512: [-100, 1300]},
               'ELG': {64: [1200, 3600], 128: [1200, 3600], 256: [1200, 3600], 512: [800, 4000]},
               'QSO': {64: [150, 450], 128: [150, 450], 256: [100, 500], 512: [0, 600]},
               }
# vrange_dict = {64: [0, 1200], 128: [-200, 1400], 256: [-600, 1800]}

min_pix_frac = 0.2  # minimum fraction of pixel area to be used

plt.rcParams['image.cmap'] = 'jet'

In [5]:
print(target_class)
target_class = target_class.lower()

maskbits = maskbits_dict[target_class.upper()]

nside = 128

npix = hp.nside2npix(nside)
pix_area = hp.pixelfunc.nside2pixarea(nside, degrees=True)
print(nside, 'Healpix size = {:.5f} sq deg'.format(pix_area))

field = 'combined'

plot_dir = os.path.join(top_plot_dir, '{}_{}_minobs_{}_maskbits_{}'.format(target_class, field, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str))
if not os.path.isdir(plot_dir):
    os.makedirs(plot_dir)

density_north = Table.read(os.path.join(target_densities_dir, 'density_map_{}_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format(target_class, 'north', nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
density_south = Table.read(os.path.join(target_densities_dir, 'density_map_{}_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format(target_class, 'south', nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
mask = (density_north['DEC']>32.375)
density_north = density_north[mask]
mask = ~np.in1d(density_south['HPXPIXEL'], density_north['HPXPIXEL'])
density = vstack([density_north, density_south[mask]])

maps = Table.read(os.path.join(randoms_counts_dir, 'counts_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format('north', nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
maps = maps[maps['n_randoms']>0]
maps_north = maps.copy()

maps = Table.read(os.path.join(randoms_counts_dir, 'counts_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format('south', nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
maps = maps[maps['n_randoms']>0]
maps_south = maps.copy()

mask = (maps_north['DEC']>32.375)
maps_north = maps_north[mask]
mask = ~np.in1d(maps_south['HPXPIXEL'], maps_north['HPXPIXEL'])
maps = vstack([maps_north, maps_south[mask]])

maps = join(maps, density[['HPXPIXEL', 'n_targets']], join_type='outer', keys='HPXPIXEL').filled(0)

print(len(maps))

area = np.sum(maps['FRACAREA'])*pix_area
print('Area = {:.1f} sq deg'.format(area))

mask = maps['FRACAREA']>min_pix_frac
maps = maps[mask]

maps['density'] = maps['n_targets'] / (pix_area * maps['FRACAREA'])

plot_path = os.path.join(plot_dir, 'density_{}_{}{}.png'.format(target_class, nside, lrgmask_str))

# if os.path.isfile(plot_path):
#     continue

map_values = np.zeros(npix)
hp_mask = np.zeros(npix, dtype=bool)
map_values[maps['HPXPIXEL']] = maps['density']
hp_mask[maps['HPXPIXEL']] = True
mplot = hp.ma(map_values)
mplot.mask = ~hp_mask

# plt.figure(figsize=(9.7, 6))
# hp.mollview(mplot, min=vrange_dict[target_class.upper()][nside][0], max=vrange_dict[target_class.upper()][nside][1],
#             rot=(120, 0, 0), fig=1, xsize=xsize_dict[nside], title='{} NSIDE={}'.format(target_class.upper(), nside))
# plt.show()

LRG
128 Healpix size = 0.20982 sq deg
96311
Area = 17980.1 sq deg


In [6]:
# healpix ras, decs
nside, nest = 128, False
pixs = np.arange(hp.nside2npix(nside))
thetas, phis = hp.pix2ang(nside, pixs, nest=nest)
ras, decs = 180. / np.pi * phis, 90. - 180. / np.pi * thetas

# mollweide settings
projection, org = "mollweide", 120

# convert to the mollweide scheme
ramws, decmws = get_radec_mw(ras, decs, org)

fig = plt.figure(figsize = (15, 7.5))
ax = fig.add_subplot(111, projection=projection)
_ = set_mwd(ax, org)
im = ax.scatter(ramws, decmws, c=mplot, s=0.25, alpha=1, rasterized=True,
          vmin=vrange_dict[target_class.upper()][nside][0], vmax=vrange_dict[target_class.upper()][nside][1])
# galactic plane
plot_gp(ax, c="0.2", lw=0.5, alpha=1, zorder=1)
#
fig.colorbar(im, orientation='horizontal', fraction=0.04, pad=0.07, aspect=25)
plt.savefig("/Users/rongpu/git/desi-targets/lrg_paper/figures/density_lrg_128_lrgmask_v1_scatter.png", bbox_inches="tight", dpi=120)
plt.close()